In [2]:
import json
import os
import sys
import argparse
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from tqdm import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    TextLoader,
    DirectoryLoader,
)
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()


True

In [3]:
BASE_DIR   = Path.cwd().parent
RAW_DIR    = BASE_DIR / "data" / "raw"
CHROMA_DIR = BASE_DIR / "data" / "chroma_db"

EMBEDDING_MODEL = "text-embedding-3-small"  

CHUNK_SIZE    = 800   # tokens ≈ characters / 4
CHUNK_OVERLAP = 150

In [4]:
COLLECTIONS: dict[str, dict] = {
    "babyos_medical": {
        "files": ["medical_reference.md"],
        "description": "Medical reference: danger signs, tests, safe foods, supplements, GDM, preeclampsia",
    },
    "babyos_germany": {
        "files": ["germany_pregnancy_guide.md"],
        "description": "Germany-specific: Mutterpass, Vorsorgeuntersuchungen, Hebamme, Elterngeld, vocabulary",
    },
    "babyos_dad": {
        "files": ["dad_partner_guide.md"],
        "description": "Dad and partner guide: trimester support, hospital bag, postpartum, admin tasks",
    },
    "babyos_faqs": {
        "files": ["pregnancy_faqs.md"],
        "description": "Pregnancy FAQs: general, trimester-specific, postpartum questions and answers",
    },
}


In [5]:
def make_embeddings() -> OpenAIEmbeddings:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not set. Add it to your .env file."
        )
    return OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=api_key)

In [6]:
def make_splitter(chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
        length_function=len,
    )


In [7]:
def load_markdown(filepath: Path, collection: str) -> list[Document]:
    """Load a markdown file and tag each chunk with metadata."""
    loader = TextLoader(str(filepath), encoding="utf-8")
    docs = loader.load()
    splitter = make_splitter()
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        chunk.metadata.update({
            "source_file": filepath.name,
            "collection":  collection,
            "chunk_index": i,
            "doc_type":    "markdown",
        })
    return chunks

In [8]:
def load_fetal_development() -> list[Document]:
    """
    Load fetal_development.json and create one rich Document per week.
    Each week becomes a single chunk — they are already the right size
    and keeping them atomic means retrieval returns complete week info.
    """
    json_path = RAW_DIR / "fetal_development.json"
    data: list[dict] = json.loads(json_path.read_text(encoding="utf-8"))

    docs = []
    for week_data in data:
        week = week_data["week"]
        trimester = week_data["trimester"]

        # Build a rich human-readable text block for embedding
        text = f"""PREGNANCY WEEK {week} — TRIMESTER {trimester}

Baby size: {week_data['size_comparison']} ({week_data['size_cm']} cm, {week_data['weight_g']} g)

FETAL DEVELOPMENT:
{week_data['development']}

BABY MILESTONES THIS WEEK:
{", ".join(week_data['baby_milestones'])}

WHAT MOM IS EXPERIENCING:
{week_data['mom_changes']}

COMMON SYMPTOMS: {", ".join(week_data['mom_symptoms'])}

FOR DAD / PARTNER:
{week_data['dad_partner_tips']}

UPCOMING APPOINTMENTS: {", ".join(week_data['appointments'])}

NUTRITION FOCUS: {", ".join(week_data['nutrition_focus'])}

DANGER SIGNS — SEEK MEDICAL HELP IMMEDIATELY IF:
{", ".join(week_data['danger_signs'])}
"""

        doc = Document(
            page_content=text,
            metadata={
                "week":         week,
                "trimester":    trimester,
                "size_cm":      week_data["size_cm"],
                "weight_g":     week_data["weight_g"],
                "size_compare": week_data["size_comparison"],
                "source_file":  "fetal_development.json",
                "collection":   "babyos_development",
                "doc_type":     "structured_json",
                "chunk_index":  week,
            }
        )
        docs.append(doc)

    return docs

In [9]:
def load_web_scraped() -> list[Document]:
    """Load all scraped .txt files from data/raw/web_scraped/."""
    web_dir = RAW_DIR / "web_scraped"
    if not web_dir.exists() or not list(web_dir.glob("*.txt")):
        print("  ⚠️  No web-scraped files found. Run corpus_fetcher.py first.")
        return []

    loader = DirectoryLoader(
        str(web_dir),
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
        show_progress=False,
    )
    docs = loader.load()
    splitter = make_splitter()
    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        fname = Path(chunk.metadata.get("source", "unknown")).name
        source_type = "nhs" if "nhs" in fname else "who" if "who" in fname else "web"
        chunk.metadata.update({
            "source_file": fname,
            "collection":  "babyos_web",
            "chunk_index": i,
            "doc_type":    "web_scraped",
            "source_type": source_type,
        })
    return chunks

In [10]:
def build_collection(
    name: str,
    docs: list[Document],
    embeddings: OpenAIEmbeddings,
    reset: bool = False,
) -> Chroma:
    """Create or update a Chroma collection from a list of Documents."""
    persist_path = str(CHROMA_DIR / name)

    if reset:
        import shutil
        if Path(persist_path).exists():
            shutil.rmtree(persist_path)
            print(f"  🗑️  Wiped existing collection: {name}")

    print(f"  📥 Indexing {len(docs)} chunks into '{name}'...")
    
    # Batch in groups of 100 to avoid rate limits
    batch_size = 100
    vectorstore = None

    for i in tqdm(range(0, len(docs), batch_size), desc=f"  {name}"):
        batch = docs[i : i + batch_size]
        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                documents=batch,
                embedding=embeddings,
                collection_name=name,
                persist_directory=persist_path,
            )
        else:
            vectorstore.add_documents(batch)

    print(f"  ✅ '{name}' — {len(docs)} chunks indexed.\n")
    return vectorstore

In [11]:
def main(reset: bool = False) -> None:
    print("=" * 60)
    print("  BabyOS RAG Ingestion Pipeline")
    print("=" * 60)

    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    embeddings = make_embeddings()

    # ── 1. Markdown collections ──────────────────────────────────────────────
    for collection_name, config in COLLECTIONS.items():
        print(f"\n[{collection_name}]")
        all_chunks: list[Document] = []
        for fname in config["files"]:
            fpath = RAW_DIR / fname
            if not fpath.exists():
                print(f"  ⚠️  File not found: {fpath} — skipping.")
                continue
            chunks = load_markdown(fpath, collection_name)
            print(f"  📄 {fname} → {len(chunks)} chunks")
            all_chunks.extend(chunks)

        if all_chunks:
            build_collection(collection_name, all_chunks, embeddings, reset)

    # ── 2. Structured JSON — fetal development ───────────────────────────────
    print("\n[babyos_development]")
    dev_docs = load_fetal_development()
    print(f"  📄 fetal_development.json → {len(dev_docs)} week documents")
    build_collection("babyos_development", dev_docs, embeddings, reset)

    # ── 3. Web-scraped content ───────────────────────────────────────────────
    print("\n[babyos_web]")
    web_docs = load_web_scraped()
    if web_docs:
        build_collection("babyos_web", web_docs, embeddings, reset)

    # ── Summary ──────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("  INGESTION COMPLETE")
    print("=" * 60)
    
    all_collections = list(COLLECTIONS.keys()) + ["babyos_development"]
    if web_docs:
        all_collections.append("babyos_web")

    total_chunks = 0
    for name in all_collections:
        persist_path = str(CHROMA_DIR / name)
        try:
            vs = Chroma(
                collection_name=name,
                embedding_function=embeddings,
                persist_directory=persist_path,
            )
            count = vs._collection.count()
            total_chunks += count
            print(f"  {name:<30} {count:>5} chunks")
        except Exception:
            print(f"  {name:<30} (not found)")

    print(f"\n  {'TOTAL':<30} {total_chunks:>5} chunks")
    print(f"\n  ChromaDB stored at: {CHROMA_DIR}\n")

In [12]:
parser = argparse.ArgumentParser(description="BabyOS RAG Ingestion")
parser.add_argument(
    "--reset",
    action="store_true",
    help="Wipe all existing collections and rebuild from scratch",
)
args, unknown = parser.parse_known_args()
main(reset=args.reset)

  BabyOS RAG Ingestion Pipeline

[babyos_medical]
  📄 medical_reference.md → 14 chunks
  📥 Indexing 14 chunks into 'babyos_medical'...


  babyos_medical: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


  ✅ 'babyos_medical' — 14 chunks indexed.


[babyos_germany]
  📄 germany_pregnancy_guide.md → 13 chunks
  📥 Indexing 13 chunks into 'babyos_germany'...


  babyos_germany: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s]


  ✅ 'babyos_germany' — 13 chunks indexed.


[babyos_dad]
  📄 dad_partner_guide.md → 19 chunks
  📥 Indexing 19 chunks into 'babyos_dad'...


...

  ✅ 'babyos_faqs' — 24 chunks indexed.


[babyos_development]
  📄 fetal_development.json → 12 week documents
  📥 Indexing 12 chunks into 'babyos_development'...


  babyos_development: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]


  ✅ 'babyos_development' — 12 chunks indexed.


[babyos_web]
  📥 Indexing 470 chunks into 'babyos_web'...


  babyos_web: 100%|██████████| 5/5 [00:03<00:00,  1.64it/s]

  ✅ 'babyos_web' — 470 chunks indexed.


  INGESTION COMPLETE
  babyos_medical                    14 chunks
  babyos_germany                    13 chunks
  babyos_dad                        19 chunks
  babyos_faqs                       24 chunks
  babyos_development                12 chunks
  babyos_web                       470 chunks

  TOTAL                            552 chunks

  ChromaDB stored at: c:\Ironhack\W8\BabyOS\data\chroma_db

